# Systematic Model Comparison Workflow

Throughout this course we have learned many forecasting methods:
naive baselines, exponential smoothing, ARIMA/SARIMA, Holt-Winters,
Prophet, and more.  How do you systematically compare them on a real
dataset and pick the best approach?

This notebook provides a **reusable workflow**:
1. Load and prepare data
2. Train/test split
3. Fit all candidate models
4. Evaluate with MAE, RMSE, MAPE
5. Build a results table
6. Forecast combination (average of top-3)
7. Decision framework for choosing a method

We use Alcohol Sales data for this exercise.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Load and Prepare Data

In [ ]:
raw = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv", parse_dates=["DATE"], index_col="DATE")
raw.index.freq = "MS"
y = raw["S4248SM144NCEN"].copy()
y.name = "AlcoholSales"

print(f"Shape: {y.shape}")
print(f"Date range: {y.index[0].date()} to {y.index[-1].date()}")
print(f"Frequency: {y.index.freq}")
y.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
y.plot(ax=ax)
ax.set_title("Monthly Alcohol Sales (S4248SM144NCEN)")
ax.set_ylabel("Millions of Dollars")
plt.tight_layout()
plt.show()

print("Upward trend with strong December spikes.")
print("Seasonal amplitude increases with level — multiplicative seasonality.")

---
## 2. Train/Test Split

Hold out the last 24 months for testing.

In [ ]:
n_test = 24
y_train = y.iloc[:-n_test]
y_test = y.iloc[-n_test:]

print(f"Train: {len(y_train)} months ({y_train.index[0].date()} to {y_train.index[-1].date()})")
print(f"Test:  {len(y_test)} months ({y_test.index[0].date()} to {y_test.index[-1].date()})")

fig, ax = plt.subplots(figsize=(14, 5))
y_train.plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
ax.axvline(y_test.index[0], color="red", linestyle="--", alpha=0.7)
ax.set_title("Train/Test Split")
ax.set_ylabel("Sales")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Helper Function: Compute Metrics

In [ ]:
def forecast_metrics(actual, predicted, model_name):
    """Compute MAE, RMSE, MAPE and return as a dict."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {"Model": model_name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)}

print("Helper function defined.")

---
## 4. Naive Baselines

Always start with baselines — any serious model must beat these.

In [ ]:
all_results = []
all_forecasts = {}

# 4a. Naive (last value)
naive_pred = np.full(n_test, y_train.iloc[-1])
all_results.append(forecast_metrics(y_test, naive_pred, "Naive"))
all_forecasts["Naive"] = naive_pred

# 4b. Seasonal Naive (repeat last year)
seasonal_naive_pred = y_train.iloc[-12:].values
seasonal_naive_pred = np.tile(seasonal_naive_pred, n_test // 12 + 1)[:n_test]
all_results.append(forecast_metrics(y_test, seasonal_naive_pred, "Seasonal Naive"))
all_forecasts["Seasonal Naive"] = seasonal_naive_pred

print("Baseline results:")
for r in all_results:
    print(f"  {r['Model']:20s}  MAE={r['MAE']:.2f}  RMSE={r['RMSE']:.2f}  MAPE={r['MAPE (%)']:.2f}%")

---
## 5. Exponential Smoothing (ETS)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Additive ETS
ets_add = ExponentialSmoothing(
    y_train, trend="add", seasonal="add", seasonal_periods=12
).fit(optimized=True)
ets_add_pred = ets_add.forecast(n_test)
all_results.append(forecast_metrics(y_test, ets_add_pred, "ETS (additive)"))
all_forecasts["ETS (additive)"] = ets_add_pred.values

# Multiplicative ETS (Holt-Winters)
ets_mul = ExponentialSmoothing(
    y_train, trend="add", seasonal="mul", seasonal_periods=12
).fit(optimized=True)
ets_mul_pred = ets_mul.forecast(n_test)
all_results.append(forecast_metrics(y_test, ets_mul_pred, "Holt-Winters (mul.)"))
all_forecasts["Holt-Winters (mul.)"] = ets_mul_pred.values

print("ETS models fitted.")

---
## 6. ARIMA / SARIMA (via pmdarima)

In [ ]:
import pmdarima as pm

# Auto ARIMA (non-seasonal)
arima_model = pm.auto_arima(
    y_train, seasonal=False, suppress_warnings=True, stepwise=True
)
arima_pred = arima_model.predict(n_periods=n_test)
all_results.append(forecast_metrics(y_test, arima_pred, "ARIMA"))
all_forecasts["ARIMA"] = arima_pred

print(f"ARIMA order: {arima_model.order}")
print(f"AIC: {arima_model.aic():.2f}")

In [ ]:
# Auto SARIMA (seasonal)
sarima_model = pm.auto_arima(
    y_train, seasonal=True, m=12, suppress_warnings=True, stepwise=True
)
sarima_pred = sarima_model.predict(n_periods=n_test)
all_results.append(forecast_metrics(y_test, sarima_pred, "SARIMA"))
all_forecasts["SARIMA"] = sarima_pred

print(f"SARIMA order: {sarima_model.order} x {sarima_model.seasonal_order}")
print(f"AIC: {sarima_model.aic():.2f}")

---
## 7. Results Table

In [ ]:
results_df = pd.DataFrame(all_results).set_index("Model").sort_values("MAE")
print("=" * 60)
print("MODEL COMPARISON — Alcohol Sales (24-month holdout)")
print("=" * 60)
results_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE (%)"]):
    results_df[metric].plot.barh(ax=ax, color="tab:blue", edgecolor="black")
    ax.set_title(metric)
    ax.invert_yaxis()

plt.suptitle("Model Comparison — Alcohol Sales", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Visual Comparison: All Forecasts vs Actuals

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# Training data (last 36 months)
y_train.iloc[-36:].plot(ax=ax, label="Train", color="tab:blue", linewidth=1)

# Test actuals
y_test.plot(ax=ax, label="Test (actual)", color="black", linewidth=2.5, marker="o", markersize=3)

# All model forecasts
colors = plt.cm.tab10(np.linspace(0, 1, len(all_forecasts)))
for (name, pred), color in zip(all_forecasts.items(), colors):
    ax.plot(y_test.index, pred, linestyle="--", color=color, label=name, alpha=0.8)

ax.set_title("All Forecasts vs Actuals — Alcohol Sales")
ax.set_ylabel("Millions of Dollars")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

---
## 9. Forecast Combination: Average of Top 3

In [ ]:
# Pick the top 3 models by MAE
top3 = results_df.index[:3].tolist()
print(f"Top 3 models: {top3}")

# Simple average of top 3
top3_preds = np.column_stack([all_forecasts[m] for m in top3])
combo_pred = top3_preds.mean(axis=1)

combo_result = forecast_metrics(y_test, combo_pred, "Combination (Top 3 Avg)")
print(f"\nCombination result:")
print(f"  MAE:  {combo_result['MAE']}")
print(f"  RMSE: {combo_result['RMSE']}")
print(f"  MAPE: {combo_result['MAPE (%)']}%")

print(f"\nBest individual ({top3[0]}):")
print(f"  MAE:  {results_df.loc[top3[0], 'MAE']}")

In [ ]:
# Plot combination
fig, ax = plt.subplots(figsize=(14, 6))
y_test.plot(ax=ax, label="Actual", color="black", linewidth=2, marker="o", markersize=3)

for name in top3:
    ax.plot(y_test.index, all_forecasts[name], linestyle=":", alpha=0.5, label=name)

ax.plot(y_test.index, combo_pred, label="Top-3 Average", color="tab:red", linewidth=2.5, linestyle="--")

ax.set_title("Forecast Combination (Top 3 Average) vs Actuals")
ax.set_ylabel("Millions of Dollars")
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Decision Framework

How to choose a forecasting method in practice:

| Question | If YES | If NO |
|----------|--------|-------|
| Is the series very short (< 2 seasons)? | Use Naive / Seasonal Naive / ETS | Continue |
| Is there a clear trend + seasonality? | Try Holt-Winters, SARIMA, Prophet | Try ARIMA, ETS |
| Is seasonality multiplicative? | Use multiplicative ETS or log-transform + additive | Use additive models |
| Do you need interpretable components? | Prophet (trend + seasonality + holidays) | SARIMA, ETS |
| Do you have external regressors? | SARIMAX, Prophet with regressors, ML models | Univariate methods |
| Do you need to forecast 1000s of series? | statsforecast (speed) | Any method |
| Do you want a unified API? | sktime | Use individual libraries |
| Are you unsure? | **Combine multiple models** | -- |

### General guidelines:
1. **Always start with baselines** (Naive, Seasonal Naive)
2. **Try 3-5 methods** from different families
3. **Use the same train/test split** for all models
4. **Evaluate with multiple metrics** (MAE, RMSE, MAPE)
5. **Consider forecast combinations** — they usually beat any single model
6. **Check residuals** of the best model for remaining patterns
7. **Retrain on all data** before making the final forecast

---
## Summary

This workflow provides a template for comparing forecasting methods:

1. **Load** data with a `DatetimeIndex` and correct frequency
2. **Split** temporally — never shuffle time series data
3. **Fit** all candidate models on the same training set
4. **Predict** the same test period with each model
5. **Evaluate** with MAE, RMSE, MAPE (and any domain-specific metric)
6. **Combine** top models — simple averaging is hard to beat
7. **Decide** using the framework above, considering accuracy + interpretability + speed

In [ ]:
print("Chapter 13 complete.")
print("Next: Chapter 14 — Capstone project (EDA + full modelling pipeline).")